# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jahnzaibakhtar/Flyrank-ml-internship-starter/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

In [ ]:
This is a scoring / ranking task, not classification. I'm not predicting a clean yes/no label
today — I'm producing a continuous priority score per content item so an editor can work down
a ranked queue. Classification would fit better once I have a genuinely observed future outcome
to predict (e.g. "did this item actually recover after review"), but for Week 1 the honest task
is ranking: which items deserve attention first.


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

In [ ]:
Target or proxy: for now I don't have a usable target — is_declining_label is a defined rule
(computed from trend_direction, which comes from trend_pct), not an observed outcome. Using it
as a label would just teach a model to reconstruct a formula I already have.

The real target I want is an observed outcome: something like "did this item's engagement/CTR
improve in the period after a review," measured in a later time window than the features. I
don't have that outcome window built yet on the starter CSV alone — it's my open task for Week 2,
likely requiring the warehouse's daily fact table to construct a proper past-features /
future-outcome split.

Until then, I'm treating this as unsupervised scoring: combining multiple decline-adjacent
signals (CTR vs. position peers, falling scroll_rate, AI-traffic dependence) into a composite
priority score, without pretending it's a trained prediction of a real future event.


## 3. Success metric

*One metric you can defend. What number means 'good'?*

In [ ]:
Success metric: precision@K on the top of the ranked queue — of the top K items flagged for
review, what fraction turn out to be genuine problems (low relative CTR/scroll/AI-traffic signal
versus peers at similar position)?

I chose precision@K over overall accuracy or AUC because the editor only ever works through the
top of the queue, not the whole dataset — the bottom of the ranking barely matters. I also want
to compare against a naive baseline (e.g. ranking by avg_position alone) so "good" means "beats
a simple sort," not just "looks plausible."

I can't fully compute this yet since I don't have an observed outcome label (see Section 2) —
what I can compute today is a proxy check: how much the top-K by my composite score differs from
top-K by avg_position alone, as a sanity signal that the score is adding information.

## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

In [ ]:
One row = one content item (pseudonymized content_id), with its trailing-90-day metrics. Below
I load the starter CSV and show the relevant slice of columns for scoring: CTR, engagement_rate,
scroll_rate, ai_traffic_pct, avg_position, and content_type (needed for the missingness flags).

In [1]:
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

cols = ["content_id", "client_id", "content_type", "avg_position",
        "ctr", "engagement_rate", "scroll_rate", "ai_traffic_pct"]

unit_of_analysis = df[cols]
print(f"Shape: {unit_of_analysis.shape[0]} content items, {unit_of_analysis.shape[1]} columns shown")
unit_of_analysis.head()

FileNotFoundError: [Errno 2] No such file or directory: 'data/raw/content_refresh_anonymized.csv'

## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

In [ ]:
A fixed if-statement rule can't handle this well for a few concrete reasons visible in the data
itself:

- Thresholds mean different things per content_type, since missingness (and likely baseline
  engagement) varies by type — one flat CTR cutoff either misses one content type or
  over-flags another.
- scroll_rate and ai_traffic_pct aren't on a normal 0-100 scale, so a rule like "flag if
  scroll_rate < 50" would misfire on items where >100 is normal for that measurement system.
- avg_position = 0 is a missing-data sentinel, not an actual top rank — any position-based rule
  has to special-case this or it silently rewards "no data" as "great position."
- The signals that matter (CTR, position, scroll, AI-traffic share) interact — a mid-position
  page with strong AI-traffic dependence needs different handling than a top-position page with
  falling scroll_rate. Writing every combination as nested if-statements becomes unmaintainable
  well before it becomes accurate.

This is exactly the situation where a scoring model earns its place: the individual signals are
each simple, but combining them correctly, per content_type, at scale, is what a human can't
hand-code cleanly.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.